In [1]:
import os
os.environ['DDEBACKEND'] = 'tensorflow'
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

import deepxde as dde 
dde.config.disable_xla_jit()

from deepxde.backend import tf

Using backend: tensorflow

2022-11-07 19:59:28.478203: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcudart.so.11.0
2022-11-07 19:59:29.717246: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcuda.so.1
2022-11-07 19:59:29.790662: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1733] Found device 0 with properties: 
pciBusID: 0000:24:00.0 name: Quadro RTX 8000 computeCapability: 7.5
coreClock: 1.77GHz coreCount: 72 deviceMemorySize: 47.46GiB deviceMemoryBandwidth: 625.94GiB/s
2022-11-07 19:59:29.790683: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcudart.so.11.0
2022-11-07 19:59:29.793013: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcublas.so.11
2022-11-07 19:59:29.793041: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dyn

Enable just-in-time compilation with XLA.

Disable just-in-time compilation with XLA.



In [5]:
"""Backend supported: tensorflow.compat.v1, tensorflow, pytorch"""
import deepxde as dde
import numpy as np

class GeometryXGeometry:
    def __init__(self, geometry, k):
        self.geometry = geometry
        self.dim = geometry.dim ** k
        self.k = k
    
    def random_points(self, n, random="pseudo"):
        xx = []
        for i in range(self.k):
            xx.append(self.geometry.random_points(n, random=random))
        return np.hstack(xx)
    
    def uniform_points(self, n, boundary=False):
        print('no uniform points')
        return self.random_points(n)
    

def heat_eq_exact_solution(x, t):
    """Returns the exact solution for a given x and t (for sinusoidal initial conditions).
    Parameters
    ----------
    x : np.ndarray
    t : np.ndarray
    """
    res = np.exp(-(n**2 * np.pi**2 * a * t) / (L**2)) * np.sin(n * np.pi * x / L)
    return res

def exact(x):
    """Returns the exact solution for a given x and t (for sinusoidal initial conditions).
    Parameters
    ----------
    x : np.ndarray
    t : np.ndarray
    """
    
    res1 = np.exp(-(n**2 * np.pi**2 * a * x[:, 1:2]) / (L**2)) * np.sin(n * np.pi * x[:, 0:1] / L)
    return res1


# Problem parameters:
a = 0.4  # Thermal diffusivity
L = 1  # Length of the bar
n = 1  # Frequency of the sinusoidal initial conditions

def pde(x, y):
    """Expresses the PDE residual of the heat equation."""
    dy_t = dde.grad.jacobian(y, x, i=0, j=1)
    dy_xx = dde.grad.hessian(y, x, i=0, j=0)
    return dy_t - a * dy_xx

def transform(x, y):
    res1 = tf.sin(np.pi * x[:, 0:1]) 
    exp1 = tf.exp(- n ** 2 * np.pi ** 2 * a * x[:, 1:2])
    
    res = res1 * (exp1 + x[:, 1:2] * y)
    #res = tf.sin(np.pi * x[:, 0:1]) * ( tf.exp(- x[:, 1:2]) + x[:, 1:2] * y)
    return res
    
# Computational geometry:
geom = dde.geometry.Interval(0, L)
timedomain = dde.geometry.TimeDomain(0, 1)
geomtime = dde.geometry.GeometryXTime(geom, timedomain)

# Initial and boundary conditions:
bc = dde.icbc.DirichletBC(geomtime, lambda x: 0, lambda _, on_boundary: on_boundary)
ic = dde.icbc.IC(
    geomtime,
    lambda x: np.sin(n * np.pi * x[:, 0:1] / L),
    lambda _, on_initial: on_initial,
)

# Define the PDE problem and configurations of the network:
data = dde.data.TimePDE(
    geomtime,
    pde,
    #[bc, ic],
    [],
    num_domain=4000,
    num_test=4000,
    solution=exact,
)

net = dde.nn.FNN([2] + [20] * 3 + [1], "tanh", "Glorot normal")
net.apply_output_transform(transform)

model = dde.Model(data, net)

# Build and train the model:
model.compile("adam", lr=1e-3, metrics=["l2 relative error"])
model.train(epochs=15000)
model.compile("L-BFGS")
losshistory, train_state = model.train()


# Plot/print the results
#dde.saveplot(losshistory, train_state, issave=True, isplot=True)
#X, y_true = gen_testdata()
#y_pred = model.predict(X)
#f = model.predict(X, operator=pde)
#print("Mean residual:", np.mean(np.absolute(f)))
#print("L2 relative error:", dde.metrics.l2_relative_error(y_true, y_pred))
#np.savetxt("test.dat", np.hstack((X, y_true, y_pred)))

Compiling model...
'compile' took 0.000357 s

Training model...

Step      Train loss    Test loss     Test metric   
0         [5.49e-01]    [5.75e-01]    [4.15e-01]    
1000      [3.12e-06]    [3.06e-06]    [3.62e-04]    
2000      [1.41e-06]    [1.42e-06]    [2.18e-04]    
3000      [6.10e-07]    [6.30e-07]    [1.30e-04]    
4000      [2.99e-07]    [3.12e-07]    [9.12e-05]    
5000      [2.33e-07]    [2.44e-07]    [1.50e-04]    
6000      [9.42e-05]    [9.76e-05]    [6.59e-03]    
7000      [5.04e-05]    [5.22e-05]    [4.82e-03]    
8000      [2.50e-06]    [2.60e-06]    [1.06e-03]    
9000      [8.03e-08]    [8.07e-08]    [5.59e-05]    
10000     [1.01e-07]    [1.03e-07]    [1.34e-04]    
11000     [9.60e-08]    [9.88e-08]    [1.33e-04]    
12000     [1.73e-05]    [1.79e-05]    [2.83e-03]    
13000     [4.31e-05]    [4.44e-05]    [4.11e-03]    
14000     [1.83e-05]    [1.88e-05]    [2.00e-03]    
15000     [6.38e-08]    [6.39e-08]    [7.45e-05]    

Best model at step 15000:
  train

In [6]:
result = np.sum(
            [np.prod(v.get_shape().as_list()) for v in model.net.trainable_variables]
        )
print('Theta:= ', result)



Theta:=  921
